# Tema 1 (Laboratorul 2) — Simularea variabilelor aleatoare

---
## Exercițiul 1 — Eșantionare aleatoare fără repetiție dintr-un fișier `.csv`

### Cerință
Scrieți un program în Python care să preia ca input un fișier `.csv` cu o listă oarecare și să aibă ca output un număr predeterminat de elemente din acea listă, fără repetiție. Aplicați pe lista studenților din grupa dumneavoastră care nu au prezentat încă o temă.

### Rezolvare
În blocul de cod de mai jos:
1. Generăm un fișier demonstrativ `studenti_neprezentati.csv` (astfel încât notebook-ul să poată fi rulat imediat, fără dependențe externe; îl puteți înlocui cu fișierul propriu al grupei).
2. Citim lista studenților din `.csv` și extragem $k$ studenți fără repetiție folosind atât `random.sample()`, cât și `numpy.random.choice(..., replace=False)`.

In [ ]:
import csv
import random
import numpy as np

# 1. Creăm un fișier .csv demonstrativ cu studenții care nu au prezentat încă tema
nume_fisier = "studenti_neprezentati.csv"
studenti_exemplu = [
    ["Nume", "Grupa"],
    ["Popescu Andrei", "A1"],
    ["Ionescu Maria", "A1"],
    ["Georgescu Vlad", "A1"],
    ["Dumitru Elena", "A1"],
    ["Stan Alexandru", "A1"],
    ["Radu Ioana", "A1"],
    ["Matei Cristian", "A1"],
    ["Munteanu Andreea", "A1"],
    ["Stoica Ștefan", "A1"],
    ["Ciobanu Diana", "A1"],
]

with open(nume_fisier, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(studenti_exemplu)

# 2. Funcție care citește fișierul .csv și extrage k elemente fără repetiție
def extrage_studenti_csv(cale_csv, k, seed=42):
    with open(cale_csv, mode="r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader, None)  # sărim peste antet
        lista_studenti = [rand[0].strip() for rand in reader if rand and rand[0].strip()]

    if k > len(lista_studenti):
        raise ValueError(f"Nu se pot extrage {k} elemente fără repetiție dintr-o listă de {len(lista_studenti)} elemente.")

    # Metoda 1: modulul standard random.sample
    random.seed(seed)
    selectie_random = random.sample(lista_studenti, k=k)

    # Metoda 2: numpy.random.choice cu replace=False
    rng = np.random.default_rng(seed)
    selectie_numpy = rng.choice(lista_studenti, size=k, replace=False).tolist()

    return lista_studenti, selectie_random, selectie_numpy

k_ales = 3
toti, sel_rnd, sel_np = extrage_studenti_csv(nume_fisier, k=k_ales, seed=42)
print(f"Total studenți eligibili în {nume_fisier}: {len(toti)}")
print(f"Selecție fără repetiție ({k_ales} studenți) cu random.sample: {sel_rnd}")
print(f"Selecție fără repetiție ({k_ales} studenți) cu numpy.choice:  {sel_np}")

---
## Exercițiul 2 — Jocul cu moneda și zarul

### Analiza teoretică
* **a) Ce fel de distribuție urmează $N$, numărul de pași ai jocului?**
  La fiecare pas, primul jucător aruncă o monedă (în mod independent) până când obține pentru prima dată **stemă** (succes, cu probabilitatea $p$). Numărul total de aruncări $N \in \{1, 2, 3, \dots\}$ necesare până la primul succes urmează o **distribuție Geometrică** de parametru $p$:
  $$P(N = n) = (1 - p)^{n - 1} p, \quad n \in \{1, 2, 3, \dots\},$$
  cu valoarea medie $\mathbb{E}[N] = \frac{1}{p}$ și dispersia $\text{Var}(N) = \frac{1 - p}{p^2}$.

* **Expresia și media teoretică a sumei $S$ (pe care al doilea jucător i-o dă primului):**
  * În primii $N - 1$ pași a picat ban, deci primul jucător i-a dat celui de-al doilea câte $0.5\$$ la fiecare pas (contribuție $-0.5(N - 1)$ pentru primul jucător).
  * La pasul $N$ a picat stemă, iar al doilea jucător aruncă zarul $Z \in \{1, 2, 3, 4, 5, 6\}$ și îi dă primului $Z - 3\$$.
  * Prin urmare, suma totală netă primită de primul jucător este:
    $$S = (Z - 3) - 0.5(N - 1).$$
  * Folosind liniaritatea mediei ($\mathbb{E}[Z] = 3.5$ și $\mathbb{E}[N] = 1/p$):
    $$\mathbb{E}[S] = (3.5 - 3) - 0.5\left(\frac{1}{p} - 1\right) = 0.5 - \frac{1 - p}{2p} = 1 - \frac{1}{2p}.$$
    * Pentru moneda cinstită ($p = 0.5$): $\mathbb{E}[S] = 1 - \frac{1}{1} = 0\$$ (joc echitabil).
    * Pentru moneda măsluită cu $p = 0.3$: $\mathbb{E}[S] = 1 - \frac{1}{0.6} = -\frac{2}{3} \approx -0.6667\$$ (avantaj pentru al doilea jucător).
    * Pentru moneda măsluită cu $p = 0.7$: $\mathbb{E}[S] = 1 - \frac{1}{1.4} = \frac{2}{7} \approx +0.2857\$$ (avantaj pentru primul jucător).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simuleaza_joc(p_stema=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    N = 0
    S = 0.0
    while True:
        N += 1
        # 1 = stemă, 0 = ban
        if rng.random() < p_stema:
            zar = int(rng.integers(1, 7))
            S += (zar - 3)
            break
        else:
            S -= 0.5
    return N, S

rng = np.random.default_rng(42)

# b) Simularea unui singur joc
N_exemplu, S_exemplu = simuleaza_joc(p_stema=0.5, rng=rng)
print(f"b) Exemplu joc (p = 0.5): N = {N_exemplu} pași, S = {S_exemplu:.2f} $")

# c) & d) Simularea a M = 100.000 jocuri pentru p = 0.5, p = 0.3 și p = 0.7
M = 100_000
valori_p = [0.5, 0.3, 0.7]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, p in zip(axes, valori_p):
    rezultate = [simuleaza_joc(p_stema=p, rng=rng) for _ in range(M)]
    N_arr = np.array([r[0] for r in rezultate])
    S_arr = np.array([r[1] for r in rezultate])

    media_S_emp = np.mean(S_arr)
    media_S_teo = 1.0 - 1.0 / (2.0 * p)
    media_N_emp = np.mean(N_arr)

    print(f"p = {p:.1f} | E[N] emp = {media_N_emp:.4f} (teo = {1/p:.4f}) | "
          f"E[S] emp = {media_S_emp:+.4f} $ (teo = {media_S_teo:+.4f} $)")

    bins = np.arange(np.floor(S_arr.min()) - 0.25, np.ceil(S_arr.max()) + 0.75, 0.5)
    ax.hist(S_arr, bins=bins, density=True, color="#457b9d", edgecolor="white", alpha=0.85)
    ax.axvline(media_S_emp, color="#e63946", linestyle="--", lw=2, label=f"Medie emp: {media_S_emp:+.3f}$")
    ax.axvline(media_S_teo, color="#1d3557", linestyle=":", lw=2, label=f"Medie teo: {media_S_teo:+.3f}$")
    ax.set_title(f"Distribuția lui S (p = {p})")
    ax.set_xlabel("Suma S ($)")
    ax.set_xlim(-8, 4)
    ax.legend(fontsize=8.5)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("Densitate empirică")
plt.tight_layout()
plt.show()

---
## Exercițiul 3 — Timpul de servire în frizerie (Mixtură de distribuții Exponențiale)

### De ce probabilitățile de preluare sunt $\frac{3}{13}, \frac{6}{13}, \frac{4}{13}$?
Cei trei frizeri lucrează în paralel cu ratele de servire $\lambda_1 = 3\text{ h}^{-1}$, $\lambda_2 = 6\text{ h}^{-1}$ și $\lambda_3 = 4\text{ h}^{-1}$. Conform proprietății **fără memorie** a distribuției exponențiale, timpii rămași până când fiecare frizer își finalizează clientul curent sunt variabile aleatoare independente $T_1 \sim \text{Exp}(\lambda_1)$, $T_2 \sim \text{Exp}(\lambda_2)$, $T_3 \sim \text{Exp}(\lambda_3)$.

Probabilitatea ca frizerul $i$ să termine primul și să preia următorul client este:
$$
P\big(T_i = \min(T_1, T_2, T_3)\big) = \frac{\lambda_i}{\lambda_1 + \lambda_2 + \lambda_3} = \frac{\lambda_i}{3 + 6 + 4} = \frac{\lambda_i}{13},
$$
adică exact $\frac{3}{13}$, $\frac{6}{13}$, respectiv $\frac{4}{13}$.

### Valori teoretice de referință pentru $X$
* **Media teoretică:**
  $$\mathbb{E}[X] = \sum_{i=1}^3 p_i \cdot \frac{1}{\lambda_i} = \frac{3}{13} \cdot \frac{1}{3} + \frac{6}{13} \cdot \frac{1}{6} + \frac{4}{13} \cdot \frac{1}{4} = \frac{3}{13} \approx 0.23077\text{ ore } (\approx 13.85\text{ minute}).$$
* **Dispersia și deviația standard teoretică:**
  $$\mathbb{E}[X^2] = \sum_{i=1}^3 p_i \cdot \frac{2}{\lambda_i^2} = \frac{3}{13}\cdot\frac{2}{9} + \frac{6}{13}\cdot\frac{2}{36} + \frac{4}{13}\cdot\frac{2}{16} = \frac{2}{13}\left(\frac{1}{3} + \frac{1}{6} + \frac{1}{4}\right) = \frac{3}{26},$$
  $$\text{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2 = \frac{3}{26} - \left(\frac{3}{13}\right)^2 = \frac{21}{338} \implies \sigma_X = \sqrt{\frac{21}{338}} \approx 0.24926\text{ ore}.$$

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N_clienti = 10_000

lambdas = np.array([3.0, 6.0, 4.0])
probabilitati = lambdas / np.sum(lambdas)  # [3/13, 6/13, 4/13]

# Alegem frizerul pentru fiecare dintre cei 10.000 de clienți
frizer_ales = rng.choice([0, 1, 2], size=N_clienti, p=probabilitati)

# Generăm timpul de servire Exp(lambda_i) => scale = 1 / lambda_i
X = rng.exponential(scale=1.0 / lambdas[frizer_ales])

media_emp = np.mean(X)
std_emp = np.std(X, ddof=1)
media_teo = 3.0 / 13.0
std_teo = np.sqrt(21.0 / 338.0)

print(f"Media empirică a lui X:            {media_emp:.5f} ore (teoretic: {media_teo:.5f} ore)")
print(f"Deviația standard empirică a lui X: {std_emp:.5f} ore (teoretic: {std_teo:.5f} ore)")

# Densitatea teoretică exactă a mixturii
x_grid = np.linspace(0, 1.5, 400)
pdf_teoretic = sum(p * stats.expon.pdf(x_grid, scale=1.0 / lam) for p, lam in zip(probabilitati, lambdas))

try:
    import arviz as az
    ax = az.plot_kde(X, label="Densitate KDE (ArviZ)", plot_kwargs={"color": "#1f77b4", "lw": 2})
    ax.plot(x_grid, pdf_teoretic, color="#d62728", linestyle="--", lw=1.8, label="Densitate teoretică exactă")
    ax.set_xlim(0, 1.5)
    ax.set_xlabel("Timp de servire X (ore)")
    ax.set_ylabel("Densitate de probabilitate")
    ax.set_title("Densitatea timpului de servire X în frizerie (N = 10.000)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    kde = stats.gaussian_kde(X)
    plt.figure(figsize=(9, 4.5))
    plt.plot(x_grid, kde(x_grid), color="#1f77b4", lw=2, label="Densitate KDE (SciPy)")
    plt.plot(x_grid, pdf_teoretic, color="#d62728", linestyle="--", lw=1.8, label="Densitate teoretică exactă")
    plt.xlim(0, 1.5)
    plt.xlabel("Timp de servire X (ore)")
    plt.ylabel("Densitate de probabilitate")
    plt.title("Densitatea timpului de servire X în frizerie (N = 10.000)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()